# SeqDiffuSeq ZH→EN + JA→EN — mBART Diagnostic Notebook

**Goal:** 25k-step diagnostic to confirm mBART-large-cc25 encoder handles Chinese/Japanese source text.  
**Why mBART:** BART-base is English-only → encoder collapses for ZH/JA source (Session 04, BLEU=0.01).  
**Change `PAIR` in Cell 1** to switch between `zh-en` and `ja-en`.

In [1]:
# ── CONFIG — change PAIR to switch language direction ─────────────────────
PAIR = "zh-en"   # "zh-en" for Chinese→English | "ja-en" for Japanese→English
# ─────────────────────────────────────────────────────────────────────────

import os, glob, re, subprocess, sys

SRC_LANG, TGT_LANG = PAIR.split("-")

# Adjust PROJECT_ROOT if running on Colab (path may differ)
PROJECT_ROOT = "/root/NER_translation"
if os.path.isdir("/content/NER_translation"):
    PROJECT_ROOT = "/content/NER_translation"

REPO_DIR   = f"{PROJECT_ROOT}/SeqDiffuSeq"
CKPT_DIR   = f"{REPO_DIR}/ckpts/{PAIR}"
DATA_DIR   = f"{REPO_DIR}/data/{PAIR}"
PRETRAINED = f"{REPO_DIR}/pretrained/mbart-large"
LOG_DIR    = f"{CKPT_DIR}/log"
OUT_DIR    = f"{CKPT_DIR}/inference_out"

for d in [CKPT_DIR, DATA_DIR, LOG_DIR, PRETRAINED, OUT_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"Pair       : {PAIR}  ({SRC_LANG} → {TGT_LANG})")
print(f"Project    : {PROJECT_ROOT}")
print(f"Repo       : {REPO_DIR}")
print(f"Checkpts   : {CKPT_DIR}")
print(f"Pretrained : {PRETRAINED}")

Pair       : zh-en  (zh → en)
Project    : /root/NER_translation
Repo       : /root/NER_translation/SeqDiffuSeq
Checkpts   : /root/NER_translation/SeqDiffuSeq/ckpts/zh-en
Pretrained : /root/NER_translation/SeqDiffuSeq/pretrained/mbart-large


In [2]:
import torch
print("CUDA     :", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU      :", torch.cuda.get_device_name(0))
    print("VRAM     :", round(props.total_memory / 1024**3, 1), "GB")
    print("PyTorch  :", torch.__version__)

CUDA     : True
GPU      : Tesla T4
VRAM     : 14.6 GB
PyTorch  : 2.6.0+cu124


## Phase 1 — Dependencies

In [3]:
import subprocess
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "bert-score", "blobfile", "datasets>=2.20",
    "huggingface-hub>=0.20",
    "mpi4py", "nltk", "numpy", "pandas", "protobuf",
    "rouge-score", "sacrebleu", "sacremoses",
    "scikit-learn", "scipy", "spacy",
    "tokenizers>=0.19", "torchmetrics", "tqdm",
    "transformers>=4.35,<4.45",
], check=True)
print("Dependencies installed.")

Dependencies installed.


## Phase 2 — Data

- **zh-en:** WMT17 zh-en (~2.1M train pairs, newstest2017 test set)
- **ja-en:** WMT20 ja-en (~3.6M train pairs)

Existing text files are automatically replaced if their line count doesn't match the HF dataset (catches stale OPUS-100 data).

In [ ]:
from datasets import load_dataset

dataset_configs = {
    "zh-en": ("wmt17", "zh-en", {"zh": "zh", "en": "en"}),
    "ja-en": ("wmt20", "ja-en", {"ja": "ja", "en": "en"}),
}

hf_dataset, hf_config, lang_keys = dataset_configs[PAIR]
print(f"==> Loading {hf_dataset} / {hf_config} ...")
ds = load_dataset(hf_dataset, hf_config, cache_dir="/root/.cache/hf")
print(f"    Splits: {list(ds.keys())}")

split_map = {"train": "train", "valid": "validation", "test": "test"}
for name, hf_split in split_map.items():
    if hf_split not in ds:
        print(f"  SKIP {name}: '{hf_split}' not in dataset")
        continue
    src_path = os.path.join(DATA_DIR, f"{name}.{SRC_LANG}")
    tgt_path = os.path.join(DATA_DIR, f"{name}.{TGT_LANG}")
    expected = len(ds[hf_split])

    if os.path.exists(src_path) and os.path.exists(tgt_path):
        n = sum(1 for _ in open(src_path, encoding="utf-8"))
        if n == expected:
            print(f"  Exists  {name}: {n:,} pairs — skipping")
            continue
        else:
            print(f"  Stale   {name}: found {n:,} but expected {expected:,} — rewriting")
            os.remove(src_path)
            os.remove(tgt_path)

    count = 0
    with open(src_path, "w", encoding="utf-8") as fs, \
         open(tgt_path, "w", encoding="utf-8") as ft:
        for ex in ds[hf_split]:
            fs.write(ex["translation"][lang_keys[SRC_LANG]].replace("\n", " ").strip() + "\n")
            ft.write(ex["translation"][lang_keys[TGT_LANG]].replace("\n", " ").strip() + "\n")
            count += 1
    print(f"  Wrote   {name}: {count:,} pairs")

# Sanity: line counts must match
for split in ["train", "valid", "test"]:
    sp = os.path.join(DATA_DIR, f"{split}.{SRC_LANG}")
    tp = os.path.join(DATA_DIR, f"{split}.{TGT_LANG}")
    if os.path.exists(sp) and os.path.exists(tp):
        ns = sum(1 for _ in open(sp, encoding="utf-8"))
        nt = sum(1 for _ in open(tp, encoding="utf-8"))
        status = "OK" if ns == nt else f"MISMATCH {ns} vs {nt}"
        print(f"  {split}: {ns:,} pairs [{status}]")
print("Data ready.")

## Phase 3 — Tokenizer (32k ByteLevelBPE)

In [ ]:
tok_vocab  = os.path.join(DATA_DIR, "vocab.json")
tok_merges = os.path.join(DATA_DIR, "merges.txt")

if os.path.exists(tok_vocab) and os.path.exists(tok_merges):
    import json
    n_vocab = len(json.load(open(tok_vocab)))
    print(f"Tokenizer already exists — {n_vocab} tokens. Skipping retraining.")
else:
    from tokenizers import ByteLevelBPETokenizer
    files = [
        os.path.join(DATA_DIR, f"train.{SRC_LANG}"),
        os.path.join(DATA_DIR, f"train.{TGT_LANG}"),
    ]
    print(f"Training 32k BPE on {[os.path.basename(f) for f in files]} ...")
    tok = ByteLevelBPETokenizer()
    tok.train(
        files=files,
        vocab_size=32000,
        min_frequency=2,
        special_tokens=["<s>", "<pad>", "</s>", "<unk>", "<mask>"],
    )
    tok.save_model(DATA_DIR)
    import json
    n_vocab = len(json.load(open(tok_vocab)))
    print(f"Tokenizer saved: {n_vocab} tokens → {DATA_DIR}")

# Sanity check
from tokenizers import ByteLevelBPETokenizer
sanity_words = {"zh-en": "你好", "ja-en": "東京"}
word = sanity_words.get(PAIR, "hello")
tok = ByteLevelBPETokenizer(tok_vocab, tok_merges)
ids = tok.encode(word).ids
print(f"Sanity: '{word}' → {len(ids)} tokens {ids}  {'OK' if len(ids) <= 6 else 'WARN'}")

Training 32k BPE on ['train.zh', 'train.en'] ...


## Phase 4 — mBART-large-cc25 Weights

~2.3 GB download — takes 3–5 min. Skipped if already cached.

In [ ]:
if os.path.exists(os.path.join(PRETRAINED, "pytorch_model.bin")):
    print("mBART already cached →", PRETRAINED)
else:
    from transformers import MBartForConditionalGeneration
    print("Downloading facebook/mbart-large-cc25 (~2.3 GB)…")
    model = MBartForConditionalGeneration.from_pretrained("facebook/mbart-large-cc25")
    model.save_pretrained(PRETRAINED, safe_serialization=False)
    del model
    print("Saved to", PRETRAINED)

## Phase 5 — Training (25k diagnostic steps)

Loss check at step 2,500:
- `< 8.0` = healthy encoder, continue
- `≈ 0.07` = collapse (same symptom as Session 04 BART-base failure)

Output streams in real-time below.

In [ ]:
# Auto-resume from latest non-EMA checkpoint
model_ckpts = sorted([
    c for c in glob.glob(os.path.join(CKPT_DIR, "model*.pt"))
    if "ema" not in os.path.basename(c)
])
resume_ckpt = model_ckpts[-1] if model_ckpts else ""
if resume_ckpt:
    m = re.search(r"model0*(\d+)\.pt", os.path.basename(resume_ckpt))
    print(f"Resuming from step {m.group(1) if m else '?'}: {os.path.basename(resume_ckpt)}")
else:
    print("Starting from scratch.")

args = [
    sys.executable, "-u", "main.py",
    "--checkpoint_path",    CKPT_DIR,
    "--src",                SRC_LANG,
    "--tgt",                TGT_LANG,
    "--train_txt_path",     f"./data/{PAIR}/train",
    "--val_txt_path",       f"./data/{PAIR}/valid",
    "--dataset",            PAIR,
    "--config_name",        PRETRAINED,
    "--diffusion_steps",    "2000",
    "--noise_schedule",     "sqrt",
    "--sequence_len",       "128",
    "--sequence_len_src",   "128",
    "--batch_size",         "32",
    "--lr",                 "1e-4",
    "--lr_anneal_steps",    "25000",
    "--warmup",             "2500",
    "--save_interval",      "5000",
    "--eval_interval",      "2500",
    "--log_interval",       "100",
    "--schedule_update_stride", "2000",
    "--loss_update_granu",  "20",
    "--schedule_sampler",   "uniform",
    "--encoder_layers",     "12",
    "--decoder_layers",     "12",
    "--num_heads",          "16",
    "--in_channel",         "1024",
    "--out_channel",        "1024",
    "--num_channels",       "4096",
    "--vocab_size",         "32005",
    "--dropout",            "0.3",
    "--predict_xstart",     "True",
    "--seed",               "42",
    "--init_pretrained",    "True",
    "--freeze_embeddings",  "False",
    "--use_pretrained_embeddings", "False",
    "--resume_checkpoint",  resume_ckpt,
]

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"]  = "0"
env["DIFFUSION_BLOB_LOGDIR"] = LOG_DIR
env["TRANSFORMERS_OFFLINE"]  = "1"

print(f"Training {PAIR} with mBART-large — 25k steps, batch=32")
print(f"Loss < 8.0 at step 2,500 = healthy | ≈ 0.07 = collapse\n")

proc = subprocess.Popen(
    args, cwd=REPO_DIR, env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()
if proc.returncode != 0:
    print(f"\nProcess exited with code {proc.returncode}")
else:
    print("\nTraining complete.")

## Phase 6 — Inference

In [ ]:
# Find latest EMA checkpoint and schedule
ema_ckpts = sorted(glob.glob(os.path.join(CKPT_DIR, "ema_0.9999_*.pt")))
if not ema_ckpts:
    raise FileNotFoundError(f"No EMA checkpoint in {CKPT_DIR} — training must reach first save_interval (5k steps)")
model_path = ema_ckpts[-1]
print(f"Checkpoint : {os.path.basename(model_path)}")

schedules = sorted(glob.glob(os.path.join(CKPT_DIR, "alpha_cumprod_step_*.npy")))
if not schedules:
    raise FileNotFoundError(f"No alpha_cumprod_step_*.npy in {CKPT_DIR}")
schedule_path = schedules[-1]
print(f"Schedule   : {os.path.basename(schedule_path)}")

test_src = os.path.join(DATA_DIR, f"test.{SRC_LANG}")
num_test = sum(1 for _ in open(test_src, encoding="utf-8"))
print(f"Test set   : {num_test:,} sentences\n")

inf_args = [
    sys.executable, "-u", "inference_main.py",
    "--model_name_or_path", model_path,
    "--val_txt_path",       f"./data/{PAIR}/test",
    "--out_dir",            OUT_DIR,
    "--time_schedule_path", schedule_path,
    "--diffusion_steps",    "2000",
    "--num_samples",        "-1",
    "--batch_size",         "50",
    "--sequence_len",       "128",
    "--sequence_len_src",   "128",
    "--top_p",              "-1",
    "--clamp",              "no_clamp",
    "--use_ddim",           "True",
    "--seed",               "42",
    "--generate_by_q",      "False",
    "--generate_by_mix",    "False",
]

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"
env["TRANSFORMERS_OFFLINE"] = "1"

proc = subprocess.Popen(
    inf_args, cwd=REPO_DIR, env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()
print(f"\nInference complete. Output in: {OUT_DIR}")

## Phase 7 — BLEU Evaluation

In [ ]:
import json, csv, sacrebleu

decoded_files = sorted([
    f for f in glob.glob(os.path.join(OUT_DIR, "ema_*.pt.samples_*.txt"))
    if "raw-output-ids" not in f
])
if not decoded_files:
    raise FileNotFoundError(f"No inference output in {OUT_DIR}")
output_file = decoded_files[-1]
print(f"Evaluating: {os.path.basename(output_file)}")

with open(output_file, "r", encoding="utf-8") as f:
    pairs = [json.loads(l.strip()) for l in f if l.strip()]
hypotheses = [p[0] for p in pairs]
references  = [p[1] for p in pairs]

# Source sentences for CSV
src_file = os.path.join(DATA_DIR, f"test.{SRC_LANG}")
sources = [l.strip() for l in open(src_file, encoding="utf-8") if l.strip()]
n = min(len(sources), len(hypotheses))
sources, hypotheses, references = sources[:n], hypotheses[:n], references[:n]

bleu_13a  = sacrebleu.corpus_bleu(hypotheses, [references], tokenize="13a")
bleu_char = sacrebleu.corpus_bleu(hypotheses, [references], tokenize="char")

m = re.search(r"ema_[\d.]+_(\d+)", os.path.basename(output_file))
step_tag = f"step{m.group(1)}" if m else "eval"
csv_path     = os.path.join(OUT_DIR, f"eval_{step_tag}.csv")
summary_path = os.path.join(OUT_DIR, f"eval_{step_tag}_summary.txt")

with open(csv_path, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([f"source_{SRC_LANG}", f"hypothesis_{TGT_LANG}", f"reference_{TGT_LANG}"])
    for src, hyp, ref in zip(sources, hypotheses, references):
        writer.writerow([src, hyp, ref])

summary = "\n".join([
    f"Pair            : {PAIR}",
    f"Output file     : {output_file}",
    f"Num samples     : {n}",
    "",
    f"SacreBLEU (13a) : {bleu_13a.score:.2f}",
    f"SacreBLEU (char): {bleu_char.score:.2f}",
])
print(summary)
with open(summary_path, "w", encoding="utf-8") as f:
    f.write(summary + "\n")
print(f"\nCSV     : {csv_path}")
print(f"Summary : {summary_path}")